In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

In [2]:
# Set cố định random seed
np.random.seed(42)
tf.random.set_seed(42)

# Tìm hiểu dữ liệu

Hãy tìm hiểu dữ liệu có trong hai file `positive_data.csv` và `negative_data.csv`

In [3]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [4]:
pos_df = pd.read_csv('/content/drive/MyDrive/CSI18/positive_data.csv')
pos_df.head(5)

,Rate,Review,label
0,9.0,Khu ẩm thực với đa dạng đồ lại còn bày trí đẹ...,1
1,9.0,Lúc nào đến aeon là lúc đấy phải tống một đốn...,1
2,10.0,Bánh ngon lại rẻ chê đâu được gần hết các loạ...,1
3,9.0,Ngon rẻ,1
4,9.6,Tôi sắp chết vì ngập trong sushi mấttttt Lên ...,1


In [5]:
neg_df = pd.read_csv('/content/drive/MyDrive/CSI18/negative_data.csv')
neg_df.head(5)

,Rate,Review,label
0,4.0,Mình thề là mình ko thể cảm nổi đồ ăn ở aeon ...,-1
1,3.8,Đôi khi thèm lên là bất chấp nắng nóng phi Và...,-1
2,3.8,Ngõ treo biển cafe trứng đúng kiểu phố cổ hà ...,-1
3,3.8,Mình thấy địa chỉ cafe Giảng ở Nguyễn Hữu Huâ...,-1
4,2.2,Mình là người Hà Nội và cũng cực kỳ khó tính ...,-1


Hãy đếm số giá trị positive và negative?

In [6]:
pos_df['label'].value_counts()

,count
label,
1,2642


In [7]:
neg_df["label"].value_counts()

,count
label,
-1,1765


In [8]:
### MỞ RỘNG ###
# Trong một số trường hợp, số lượng samples của các class có tỉ lệ quá khác biệt sẽ ảnh hưởng đến chất lượng của mô hình
# Vì vậy nên sử dụng DataFrame.sample() để cho số lượng positive bằng với số lượng negative
pos_df_sampled = pos_df.sample(n=len(neg_df), random_state=42)
pos_df_sampled['label'].value_counts()

,count
label,
1,1765


Kết hợp dữ liệu positive và negative vào cùng một DataFrame bằng `pandas.concat`

In [9]:
df = pd.concat([pos_df_sampled, neg_df])
df['label'].value_counts()

,count
label,
1,1765
-1,1765


In [10]:
# Vì sử dụng phương thức pandas.concat() nên các sample positive và negative hiện đang không được phân bố đồng đều trong DataFrame
df.tail(10)

,Rate,Review,label
1755,1.0,Mình du lịch ra ở khách sạn search trên foody...,-1
1756,3.0,Chả hiểu sao có người review rộng rãi và sạch...,-1
1757,3.4,Thấy khen nhiều nên lặn lội đến ăn Ko quán rấ...,-1
1758,3.4,Nghe mn bảo ngon nên đến ăn thử nhưng hãi Thị...,-1
1759,2.6,Nước chấm pate ko khoai tây chiên chán nói ch...,-1
1760,3.2,Mình đã ăn bò bít tết tại bò dai và chán kinh...,-1
1761,3.2,Ăn không Sốt ít mà khoai mềm được mỗi quả trứ...,-1
1762,2.8,Sự thực là khá tò mò khi thấy bạn bè check in...,-1
1763,3.8,Vị nằm ngay mặt đường nên dễ nhưng gửi xe lại...,-1
1764,4.0,mình uống ở cơ sở thụy trên đg thụy khuê có c...,-1


Để phân phối lại các sample positive và negative trong df, sử dụng phương thức [DataFrame.sample](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html).

Ở đây ta chọn `frac=1` để chọn **tất cả** các sample trong df với thứ tự ngẫu nhiên.

In [11]:
df = df.sample(frac=1).reset_index(drop=True)
df.head()

,Rate,Review,label
0,9.0,Nhà mình người đi ăn mà hóa đơn là đắt hay rẻ...,1
1,3.4,Mình nói thật mình ăn ở đây nhiều rồi nhưng h...,-1
2,2.8,Đặc sệt hơi hướng nhà Nhân viên thì đông như ...,-1
3,9.0,Đến lần k phải giờ cao điểm nên khá thoải mái...,1
4,9.4,Soya bean ăn chất lượng,1


Đầu ra của mô hình phân loại hai lớp (binary classification) là giá trị 0 hoặc 1.

Vì vậy cần chuyển các giá trị -1 thành 0 bằng [DataFrame.replace](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.replace.html).

In [12]:
# DataFrame.replace() nhận vào một dictionary, với key là giá trị cần thay đổi và value là giá trị mới
df['label'] = df['label'].replace({-1:0})
df['label'].value_counts()

,count
label,
1,1765
0,1765


# Chia tập train - test


Sử dụng thuộc tính `values` của Pandas Series để lấy các giá trị dưới dạng **numpy ndarray**.

Các dữ liệu dùng để train và test nên được chuyển đổi sang kiểu `ndarray` để phù hợp với các hàm tính toán trong Machine Learning và Deep Learning.

In [13]:
X = df['Review'].values
y = df['label'].values
X.shape, y.shape

((3530,), (3530,))

Sử dụng hàm `train_test_split` của sklearn để chia tập train và test theo tỉ lệ 8:2.

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = np.array(X_train)
X_test = np.array(X_test)
y_train = np.array(y_train)
y_test = np.array(y_test)
X_train.shape, X_test.shape

((2824,), (706,))

# Tokenize dữ liệu train - test
Vận dụng các kiến thức tokenizing và padding ở buổi trước để xử lý dữ liệu chữ.\
Nhắc lại các bước xử lý:
1. Chọn kích thước từ điển
2. Chọn độ dài lớn nhất của một chuỗi (sequence)
3. Tạo Tokenizer và fit Tokenizer (trên văn bản của tập train)
4. Sử dụng Tokenizer đã huấn luyện để tạo tokens
5. Sử dụng padding và truncating để các chuỗi có độ dài bằng nhau

In [16]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences

In [17]:
# Chọn kích thước từ điển là 10000
# và độ dài lớn nhất của một bình luận là 400 tokens
vocab_size = 10000
max_length = 400

# Tạo tokenizer và fit trên tập train
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<00V>")
tokenizer.fit_on_texts(X_train)

# Sử dụng tokenizer đã fit để tokenize các bình luận trên tập train
X_train_seqs = tokenizer.texts_to_sequences(X_train)
X_train_padded = pad_sequences(X_train_seqs, maxlen=max_length, padding='post', truncating='post') # thực hiện padding

# Sử dụng tokenizer đã fit để tokenize các bình luận trên tập test
X_test_seqs = tokenizer.texts_to_sequences(X_test)
X_test_padded = pad_sequences(X_test_seqs, maxlen=max_length, padding='post', truncating='post')

print("Length of train dataset:", len(X_train_padded))
print("Length of test dataset:", len(X_test_padded))

Length of train dataset: 2824
Length of test dataset: 706


# Định nghĩa mô hình NLP

Mô hình phân loại ngôn ngữ đơn giản gồm 2 phần:
1. Layer Embedding: Để học ngôn ngữ
2. Block phân loại: Tương tự với mô hình phân loại hình ảnh

Vì mục tiêu bài toán là phân loại 02 lớp nên:
* Layer Dense cuối cùng có 1 unit và có activation là `sigmoid`
* Compile với loss là `binary_crossentropy` và metric là `accuracy`
* Mặc định sử dụng optimizer là `adam`.

In [18]:
# Chọn chiều embedding là 128
embedding_dim = 128

model = keras.Sequential([
    # Layer Embedding
    keras.layers.Embedding(vocab_size, embedding_dim, trainable=True),
    # Block phân loại (đã học)
     keras.layers.Flatten(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid'),
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Sử dụng thêm ModelCheckpoint và chạy phương thức `fit` để huấn luyện mô hình với các tham số sau:
* Tập train là `X_train` và `y_train`
* Tập validation là `X_test` và `y_test`
* Số lượng epochs là `5`

In [19]:
from tensorflow.keras.callbacks import ModelCheckpoint

# Định nghĩa model checkpoint
checkpoint = ModelCheckpoint(
    filepath='model_epoch_{epoch:02d}.keras',
    save_weights_only=False,
    save_best_only=False,
    monitor='val_loss',
    verbose=1
)

# Huấn luyện mô hình với config đã chọn
history = model.fit(
    X_train_padded, y_train,
    validation_data=(X_test_padded, y_test),
    epochs=5,               # Số lần train toàn bộ dữ liệu
    callbacks=[checkpoint]               # Thêm config checkpoint
)

Epoch 1/5
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.5941 - loss: 0.6950
Epoch 1: saving model to model_epoch_01.keras

Epoch 1: finished saving model to model_epoch_01.keras
89/89 ━━━━━━━━━━━━━━━━━━━━ 8s 77ms/step - accuracy: 0.6969 - loss: 0.5650 - val_accuracy: 0.8640 - val_loss: 0.3288
Epoch 2/5
88/89 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.9255 - loss: 0.1982
Epoch 2: saving model to model_epoch_02.keras

Epoch 2: finished saving model to model_epoch_02.keras
89/89 ━━━━━━━━━━━━━━━━━━━━ 9s 66ms/step - accuracy: 0.9448 - loss: 0.1607 - val_accuracy: 0.8754 - val_loss: 0.3141
Epoch 3/5
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.9888 - loss: 0.0556
Epoch 3: saving model to model_epoch_03.keras

Epoch 3: finished saving model to model_epoch_03.keras
89/89 ━━━━━━━━━━━━━━━━━━━━ 9s 99ms/step - accuracy: 0.9897 - loss: 0.0504 - val_accuracy: 0.8754 - val_loss: 0.3417
Epoch 4/5
88/89 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.9955 - loss: 0.0243
Epoch 4: s

# Sử dụng mô hình
Hãy tạo một số dữ liệu mẫu để đánh giá mô hình.\
Ở đây ta có một số bình luận mẫu `test_feedback` và giá trị phân loại tương ứng `test_truth`.

In [22]:
test_reviews = [
    "Thịt bị hôi. Nước bún thì nhạt. Đề nghị đừng ăn.",
    "Đồ ăn k giống review trên mạng, chất lượng tệ, không ngon. Tốn tiền!",
    "Rất thích không gian quán",
    "Không thể tin được phải đi 5km để ăn một món như thế này",
    "Không thể tin được chỉ cần đi 5km để ăn một món như thế này",
    "Tôi không ghét món dưa leo lắm",
    "Tôi không ghét món dưa leo lắm. Ăn cũng được",
    "Đồ ăn nhiều bột ngọt. Không ăn nữa",
    "Nhiều đồ ăn, khá ngon. Nhưng giá cả hơi mắc. Tuy nhiên không gian quán đẹp, sẽ quay lại lần sau"
    ]
test_gt = [0, 0, 1, 0, 1, 1, 1, 0, 1]
test_df = pd.DataFrame({"Review": test_reviews, "label": test_gt})
test_df

,Review,label
0,Thịt bị hôi. Nước bún thì nhạt. Đề nghị đừng ăn.,0
1,"Đồ ăn k giống review trên mạng, chất lượng tệ,...",0
2,Rất thích không gian quán,1
3,Không thể tin được phải đi 5km để ăn một món n...,0
4,Không thể tin được chỉ cần đi 5km để ăn một mó...,1
5,Tôi không ghét món dưa leo lắm,1
6,Tôi không ghét món dưa leo lắm. Ăn cũng được,1
7,Đồ ăn nhiều bột ngọt. Không ăn nữa,0
8,"Nhiều đồ ăn, khá ngon. Nhưng giá cả hơi mắc. T...",1


Các bước sử dụng mô hình để phân loại:
1. Sử dụng tokenizer đã fit ở bước trước để tokenize các bình luận
2. Thực hiện padding tương ứng
3. Dự đoán bằng phương thức `predict` với mô hình
4. Xử lý kết quả dự đoán

In [23]:
# Sử dụng lại tokenizer
test_reviews = test_df['Review'].values
test_labels = test_df['label'].values
test_seqs = tokenizer.texts_to_sequences(test_reviews)
test_padded = pad_sequences(test_seqs, padding='post', truncating='post', maxlen=max_length)

Load checkpoint với chỉ số tốt nhất và sử dụng phương thức `predict` để dự đoán.

In [25]:
from tensorflow.keras.models import load_model

loaded_model = load_model("model_epoch_03.keras")
preds = loaded_model.predict(test_padded)
preds

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step


array([[0.02408403],
       [0.04310931],
       [0.99365854],
       [0.2507152 ],
       [0.11863186],
       [0.61494446],
       [0.59873885],
       [0.51493204],
       [0.8031016 ]], dtype=float32)

In [26]:
classes = {"1":"positive", "0": "negative"}
acc = 0
i = 0
threshold = 0.5

for fb, score in zip(test_reviews, preds):
    prediction = (score>threshold).astype(int)
    print(fb, "|", classes[str(test_labels[i])], "predicted as", classes[str(prediction[0])], "with score =", score[0])

    if prediction == test_labels[i]:
        acc += 1
    i += 1

print("\nAccuracy:", round(acc/len(test_reviews), 2))

Thịt bị hôi. Nước bún thì nhạt. Đề nghị đừng ăn. | negative predicted as negative with score = 0.024084028
Đồ ăn k giống review trên mạng, chất lượng tệ, không ngon. Tốn tiền! | negative predicted as negative with score = 0.043109305
Rất thích không gian quán | positive predicted as positive with score = 0.99365854
Không thể tin được phải đi 5km để ăn một món như thế này | negative predicted as negative with score = 0.2507152
Không thể tin được chỉ cần đi 5km để ăn một món như thế này | positive predicted as negative with score = 0.11863186
Tôi không ghét món dưa leo lắm | positive predicted as positive with score = 0.61494446
Tôi không ghét món dưa leo lắm. Ăn cũng được | positive predicted as positive with score = 0.59873885
Đồ ăn nhiều bột ngọt. Không ăn nữa | negative predicted as positive with score = 0.51493204
Nhiều đồ ăn, khá ngon. Nhưng giá cả hơi mắc. Tuy nhiên không gian quán đẹp, sẽ quay lại lần sau | positive predicted as positive with score = 0.8031016

Accuracy: 0.78
